In [1]:
import os
import cv2
import pandas as pd
from tqdm import tqdm

In [2]:
def apply_ben_graham(img, target_size=224):
    """
    Aplică DOAR procesarea Ben Graham (corectarea iluminării + evidențiere leziuni).
    Redimensionează imaginea la 224x224.
    """
    img = cv2.resize(img, (target_size, target_size))
    blur = cv2.GaussianBlur(img, (0, 0), sigmaX=10)
    # Combinația magică: adaugă fundalul gri (128) și scoate leziunile în evidență
    img_processed = cv2.addWeighted(img, 4, blur, -4, 128)
    return img_processed

def get_real_image_path(base_dir, img_name_from_csv):
    """Caută fizic pe disc fișierul, ignorând extensia greșită din CSV."""
    nume_baza = os.path.splitext(img_name_from_csv)[0] 
    extensii_posibile = ['.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG']
    
    for ext in extensii_posibile:
        cale_test = os.path.join(base_dir, nume_baza + ext)
        if os.path.exists(cale_test):
            return cale_test
            
    return None

In [7]:
def main():
    root_dir = r'B:\Projects\Disertatie\Diabetic-Retinopathy-Classifier'
    # Creăm un folder nou, separat de cel cu augmentări
    output_dir = os.path.join(root_dir, 'datasets_bengraham')
    image_size = 224

    # Configurație actualizată pentru datele preprocesate (datasets_augmented)
    datasets_config = {
        'aptos': {
            'train': {'csv': 'datasets_augmented/aptos/train/aptos_train_balanced.csv', 
                      'img': 'datasets_augmented/aptos/train/images'},
            'test':  {'csv': 'datasets_augmented/aptos/test/aptos_test_processed.csv',  
                      'img': 'datasets_augmented/aptos/test/images'}
        },
        'eyepacs': {
            'train': {'csv': 'datasets_augmented/eyepacs/train/eyepacs_train_balanced.csv', 
                      'img': 'datasets_augmented/eyepacs/train/images'},
            'test':  {'csv': 'datasets_augmented/eyepacs/test/eyepacs_test_processed.csv',  
                      'img': 'datasets_augmented/eyepacs/test/images'}
        },
        'idrid': {
            'train': {'csv': 'datasets_augmented/idrid/train/idrid_train_balanced.csv', 
                      'img': 'datasets_augmented/idrid/train/images'},
            'test':  {'csv': 'datasets_augmented/idrid/test/idrid_test_processed.csv', 
                      'img': 'datasets_augmented/idrid/test/images'}
        },
        'messidor2': {
            'train': {'csv': 'datasets_augmented/messidor2/train/messidor2_train_balanced.csv', 
                      'img': 'datasets_augmented/messidor2/train/images'},
            'test':  {'csv': 'datasets_augmented/messidor2/test/messidor2_test_processed.csv', 
                      'img': 'datasets_augmented/messidor2/test/images'} 
        }
    }

    print("🚀 Începem preprocesarea (DOAR Ben Graham, distribuție 1:1 originală)...\n")

    for dataset_name, splits in datasets_config.items():
        print(f"=========================================")
        print(f" Procesare Dataset: {dataset_name.upper()}")
        print(f"=========================================")
        
        for split_name, config in splits.items():
            csv_path = os.path.join(root_dir, config['csv'])
            img_in_dir = os.path.join(root_dir, config['img'])
            
            try:
                df = pd.read_csv(csv_path)
            except FileNotFoundError:
                print(f"[Eroare] Nu am găsit {csv_path}. Sărim peste.")
                continue
            
            save_path = os.path.join(output_dir, dataset_name, split_name, 'images')
            os.makedirs(save_path, exist_ok=True)
            
            new_data = [] 
            
            print(f"[{split_name.upper()}] Păstrăm numărul de imagini exact ca în setul original.")
            
            # De data aceasta, nu mai calculăm "max_samples", ci parcurgem pur și simplu tot CSV-ul rând cu rând
            pbar = tqdm(total=len(df), desc=f"  Procesare {split_name}", ncols=90)
            
            for idx in range(len(df)):
                # Folosim .iloc[rand, coloana] pentru a ignora complet numele coloanelor
                orig_name = str(df.iloc[idx, 0]) 
                class_id = int(df.iloc[idx, 1])
                
                nume_curat = os.path.splitext(orig_name)[0]
                # Nume simplu, doar pentru a păstra clasa în vizibilitate
                new_filename = f"c{class_id}_{nume_curat}.jpg"
                output_path = os.path.join(save_path, new_filename)
                
                # --- LOGICA DE RESUME ---
                if os.path.exists(output_path):
                    new_data.append([new_filename, class_id])
                    pbar.update(1)
                    continue
                
                img_path = get_real_image_path(img_in_dir, orig_name)
                
                if img_path is None:
                    pbar.update(1)
                    continue
                    
                img = cv2.imread(img_path)
                if img is None:
                    pbar.update(1)
                    continue
                    
                # Aplicăm magia Ben Graham
                img_processed = apply_ben_graham(img, image_size)
                
                # Salvăm
                cv2.imwrite(output_path, img_processed)
                new_data.append([new_filename, class_id])
                
                pbar.update(1)
                
            pbar.close()

            # --- SALVĂM NOUL CSV ---
            new_df = pd.DataFrame(new_data, columns=['id_code', 'diagnosis'])
            
            # Nume sugestiv pentru noul CSV
            nume_csv_nou = f"{dataset_name}_{split_name}_bengraham.csv"
            cale_csv_nou = os.path.join(output_dir, dataset_name, split_name, nume_csv_nou)
            
            # Amestecăm doar train-ul pentru stabilitatea DataLoader-ului în timpul antrenamentului
            if split_name == 'train':
                new_df = new_df.sample(frac=1).reset_index(drop=True)
                
            new_df.to_csv(cale_csv_nou, index=False)
            
    print("\n✅ PROCESARE FINALIZATĂ! Folderul 'datasets_bengraham' este gata de explorat.")

In [8]:
if __name__ == '__main__':
    main()  

🚀 Începem preprocesarea (DOAR Ben Graham, distribuție 1:1 originală)...

 Procesare Dataset: APTOS
[TRAIN] Păstrăm numărul de imagini exact ca în setul original.


  Procesare train: 100%|█████████████████████████████| 7170/7170 [00:23<00:00, 309.16it/s]


[TEST] Păstrăm numărul de imagini exact ca în setul original.


  Procesare test: 100%|████████████████████████████████| 366/366 [00:01<00:00, 274.28it/s]


 Procesare Dataset: EYEPACS
[TRAIN] Păstrăm numărul de imagini exact ca în setul original.


...

 Procesare Dataset: MESSIDOR2
[TRAIN] Păstrăm numărul de imagini exact ca în setul original.


  Procesare train: 100%|██████████████████████████████| 5085/5085 [00:57<00:00, 89.02it/s]


[TEST] Păstrăm numărul de imagini exact ca în setul original.


  Procesare test: 100%|███████████████████████████████| 1744/1744 [00:22<00:00, 78.27it/s]


✅ PROCESARE FINALIZATĂ! Folderul 'datasets_bengraham' este gata de explorat.


In [9]:
# print number of images in each class for each dataset and split
for dataset_name in ['aptos', 'eyepacs', 'idrid', 'messidor2']:
    for split in ['train', 'test']:
        csv_path = f'../../datasets_bengraham/{dataset_name}/{split}/{dataset_name}_{split}_bengraham.csv'
        try:
            df = pd.read_csv(csv_path)
            print(f"{dataset_name} - {split}:")
            print(df['diagnosis'].value_counts())
            print()
        except FileNotFoundError:
            print(f"Nu am găsit {csv_path}. Sărim peste.")

aptos - train:
diagnosis
0    1434
3    1434
4    1434
1    1434
2    1434
Name: count, dtype: int64

aptos - test:
diagnosis
0    199
2     87
4     33
1     30
3     17
Name: count, dtype: int64

eyepacs - train:
diagnosis
4    25810
3    25810
2    25810
1    25810
0    25810
Name: count, dtype: int64

eyepacs - test:
diagnosis
0    39533
2     7861
1     3762
3     1214
4     1206
Name: count, dtype: int64

idrid - train:
diagnosis
3    156
1    156
4    156
2    156
0    156
Name: count, dtype: int64

idrid - test:
diagnosis
2    156
0    129
3     84
4     64
1     22
Name: count, dtype: int64

messidor2 - train:
diagnosis
0    1017
2    1017
3    1017
4    1017
1    1017
Name: count, dtype: int64

messidor2 - test:
diagnosis
0    1017
2     347
1     270
3      75
4      35
Name: count, dtype: int64

